In [1]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import boto3
import os
import glob
import pandas as pd
import numpy as np
import h5py
from scipy.signal import butter, filtfilt
from sklearn.model_selection import KFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


**UPLOADING INTO S3**

In [3]:
bucket_name = "hpc-dataset"        # Name of the S3 bucket you want to create/use
s3 = boto3.client("s3")            # Create an S3 client using boto3
region = boto3.session.Session().region_name  # Get the current AWS region of your session

# --- Create bucket safely ---
try:
    if region == "us-east-1":   # Special case: buckets in 'us-east-1' don’t need LocationConstraint
        s3.create_bucket(Bucket=bucket_name)  # Create the bucket in 'us-east-1'
    else:
        s3.create_bucket(                     # Create the bucket in the specified region
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region}
        )
    print(f"Bucket {bucket_name} created in {region}")  # Confirmation message
except s3.exceptions.BucketAlreadyOwnedByYou:  
    # If bucket name already exists and is owned by you
    print(f"Bucket {bucket_name} already exists and is owned by you")
except s3.exceptions.BucketAlreadyExists:      
    # If bucket name exists globally (S3 bucket names are unique across all AWS users)
    print(f"Bucket {bucket_name} already exists globally, choose another name")
except Exception as e:                        
    # Catch any other errors in bucket creation
    print("Bucket creation failed:", e)

try:
    s3.head_bucket(Bucket=bucket_name)  # Check if the bucket is accessible/exists
    print("Bucket confirmed, starting upload...")

    # Upload a CSV file to the 'raw' folder in the bucket
    s3.upload_file("preprocess_metadata.csv", bucket_name, "raw/preprocess_metadata.csv")

    # Upload all .hdf5 files in the current directory to 'raw/' folder in S3
    for f in glob.glob("*.hdf5"):
        s3.upload_file(f, bucket_name, f"raw/{os.path.basename(f)}")

    print("Dataset uploaded to S3 successfully!")  # Confirmation of successful upload

except botocore.exceptions.ClientError as e:
    # Catch AWS client errors (e.g., permission issues, wrong bucket name, etc.)
    print("Upload failed:", e)


Bucket hpc-dataset already exists and is owned by you
Bucket confirmed, starting upload...
Dataset uploaded to S3 successfully!


**PREPROCESSING THE HDF5 FILE**

In [ ]:

# ====================== PREPROCESSING UTILITIES ======================
def bandpass_filter(signal, lowcut=20, highcut=450, fs=1000, order=4):
    """
    Apply a Butterworth bandpass filter to EMG signals.
    butterworth bandpass filter : A Butterworth bandpass filter is a signal filter that passes only the frequencies within a specified range while blocking others.
    It is designed to have a maximally flat frequency response (no ripples) in the passband.
    Keeps frequencies between lowcut and highcut Hz.
    """
    nyq = 0.5 * fs  # Nyquist frequency
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, signal, axis=0)  # Zero-phase filtering (no phase distoration)


def preprocess_emg(emg, fs=1000):
    """
    Preprocess EMG data:
    1. Bandpass filter
    2. Rectify (absolute value so that we can identify the signal intensity)
    3. RMS smoothing with sliding window
    4. Normalize (zero mean, unit variance)
    """
    emg_filtered = bandpass_filter(emg, fs=fs)
    emg_rect = np.abs(emg_filtered)
    window = 50
    emg_processed = np.zeros_like(emg_rect) #zero array with size of emg rect

    # RMS smoothing per channel, RMS envelope of each EMG channel using a moving average window.
    for i in range(emg_rect.shape[1]):
        emg_processed[:, i] = np.sqrt(
            np.convolve(emg_rect[:, i]**2, np.ones(window)/window, mode='same')
        )

    # Normalize per channel
    return (emg_processed - emg_processed.mean(axis=0)) / (emg_processed.std(axis=0)+1e-8)


def preprocess_angles(angles):
    """
    Normalize joint angle data (zero mean, unit variance).
    """
    return (angles - np.mean(angles, axis=0)) / (np.std(angles, axis=0)+1e-8)


# ====================== PROCESSING LOOP ======================
meta = pd.read_csv("preprocess_metadata.csv")  # Load metadata file
all_segments = []   # Store processed DataFrames
failed_files = []   # Track missing or failed files

print(f"Loaded metadata with {len(meta)} entries")

# Iterate over each metadata entry
for idx, row in meta.iterrows():
    fname = row['filename']
    start, end = row['start'], row['end']
    stage = row['stage']

    print(f"\nProcessing {idx+1}/{len(meta)} : {fname}")

    # --- Check if file exists ---
    if not os.path.exists(fname):
        print(f"Missing file: {fname}")
        failed_files.append(fname)
        continue

    try:
        # Open HDF5 file
        with h5py.File(fname, 'r') as f:
            # --- Check dataset presence ---
            if "emg2pose/timeseries" not in f:
                print(f"Skipping {fname}, dataset 'emg2pose/timeseries' not found")
                failed_files.append(fname)
                continue
                
            # --- Load structured dataset ---
            timeseries_data = f["emg2pose/timeseries"]
            print(f"Dataset shape: {timeseries_data.shape}")
            print(f"Dataset fields: {timeseries_data.dtype.names}")
            
            # Extract fields (time, EMG, joint angles)
            time = timeseries_data['time'][:]              # shape (N,)
            emg = timeseries_data['emg'][:]                # shape (N, 16)
            angles = timeseries_data['joint_angles'][:]    # shape (N, 20)
            
            print(f"Data shapes - Time: {time.shape}, EMG: {emg.shape}, Angles: {angles.shape}")

        # --- Select samples within metadata time range ---
        mask = (time >= start) & (time <= end)
        if mask.sum() == 0:
            print(f"No samples in range [{start}, {end}] for {fname}")
            continue

        print(f"Found {mask.sum()} samples in time range [{start}, {end}]")

        # Filtered data
        emg_filtered = emg[mask]
        angles_filtered = angles[mask]
        time_filtered = time[mask]
        
        # Apply preprocessing
        emg_processed = preprocess_emg(emg_filtered)
        angles_processed = preprocess_angles(angles_filtered)

        # --- Create DataFrame with processed data ---
        df = pd.DataFrame()
        
        # Add EMG features
        for i in range(emg_processed.shape[1]):
            df[f'emg_{i}'] = emg_processed[:, i]
        
        # Add angle features
        for j in range(angles_processed.shape[1]):
            df[f'angle_{j}'] = angles_processed[:, j]

        # Add metadata fields
        df['time'] = time_filtered
        df['stage'] = stage
        df['filename'] = fname
        
        df['session'] = row.get('session', 'unknown')
        df['user'] = row.get('user', 'unknown') 
        df['side'] = row.get('side', 'unknown')

        # Store processed DataFrame
        all_segments.append(df)
        print(f"Successfully processed {len(df)} samples")

    except Exception as e:
        print(f"Error processing {fname}: {str(e)}")
        failed_files.append(fname)
        import traceback
        traceback.print_exc()


# ====================== SAVE FINAL DATASET ======================
if all_segments:
    # Combine all processed DataFrames
    processed_df = pd.concat(all_segments, ignore_index=True)

    # Save to CSV
    processed_df.to_csv("processed_dataset.csv", index=False)

    # Upload to S3
    s3.upload_file("processed_dataset.csv", bucket_name, "processed/processed_dataset.csv")

    print(f"\nPreprocessing complete! Final dataset shape: {processed_df.shape}")
    print(f"Columns: {list(processed_df.columns)}")
    print(f"Stages: {processed_df['stage'].value_counts().to_dict()}")
else:
    print("No data processed successfully")


# ====================== SUMMARY ======================
summary = {
    "total_entries": len(meta),
    "processed_segments": len(all_segments),
    "successful_files": len(set([df['filename'].iloc[0] for df in all_segments])),
    "failed_files_count": len(failed_files),
    "failed_files": failed_files,
    "final_dataset_shape": processed_df.shape if all_segments else (0, 0)
}

# Save and upload summary
summary_df = pd.DataFrame([summary])
summary_df.to_csv("processing_summary.csv", index=False)
s3.upload_file("processing_summary.csv", bucket_name, "processed/processing_summary.csv")
print("Summary uploaded to S3")


# --- Display final summary ---
print(f"\nPROCESSING SUMMARY:")
print(f"Total metadata entries: {summary['total_entries']}")
print(f"Successfully processed segments: {summary['processed_segments']}")
print(f"Unique files processed: {summary['successful_files']}")
print(f"Failed files: {summary['failed_files_count']}")
if summary['failed_files_count'] > 0:
    print(f"Failed file list: {summary['failed_files'][:5]}...")


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Bucket hpc-dataset already exists and is owned by you
Bucket confirmed, starting upload...
Raw dataset uploaded to S3
Loaded metadata with 30 entries

Processing 1/30 : 2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-1_left.hdf5
Dataset shape: (142674,)
Dataset fields: ('time', 'joint_angles', 'emg')
Data shapes - Time: (142674,), EMG: (142674, 16), Angles: (142674, 20)
Found 142003 samples in time range [1670312562, 1670312633]
Successfully processed 142003 samples

Processing 2/30 : 2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-1_right.hdf5
Dataset shape: (142652,)
Dataset fields: ('time', 'joint_angles', 'emg')
Data shapes - Time: (142652,), EMG: (142652, 16), Angles: (142652, 20)
Found 141997 samples in time range [1670312562, 1670312633]
Successfully processed 141997 samples

Processing 3/30 : 2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-10_left.hdf5
Dataset shape: (119266,)
Dataset fields: ('time', 'joint_angles', 'emg')
Data shapes - Time: (119